# Sample

This notebook goes with `vak`. Each section takes one idea at its smallest size, small enough to check by hand.
The matching command then runs the same functions on the pair counts from 50,000 stories.

Before this: step 4 counts pairs, the smallest language model. After this: step 6 starts on the transformer, with
one attention head.

| Section | Idea | Used in |
|---|---|---|
| 1 | Sample: temperature, top-k, top-p, and the decode loop | `vak write` |

## The toy model from step 4

Sampling needs a model. This cell rebuilds the toy from `04-counts.ipynb` with the same function. It has six
tokens and one story to learn from: "the cat sat on the mat", with the end token before and after it.

In [1]:
from vak.counts import PairCounts

words = ["<|endoftext|>", "the", "cat", "sat", "on", "mat"]  # a token's id is its place in this list
story = []
for word in ["<|endoftext|>", "the", "cat", "sat", "on", "the", "mat", "<|endoftext|>"]:
    story.append(words.index(word))
toy = PairCounts.from_stories([story], vocab_size=len(words))
print("the story as ids:", story)
print('counts after "the":', toy.counts[1].tolist())

the story as ids: [0, 1, 2, 3, 4, 1, 5, 0]
counts after "the": [0.0, 0.0, 1.0, 0.0, 0.0, 1.0]


**What you see**

The story is `[0, 1, 2, 3, 4, 1, 5, 0]`, as in step 4. In training, cat and mat each follow "the" once, so the
counts after "the" are `[0.0, 0.0, 1.0, 0.0, 0.0, 1.0]`. Section 1 turns this row into chances.

## 1. Sample: temperature, top-k and top-p

A model gives one score, a **logit**, to every token. Softmax turns the scores into chances. The **sampler** then
reshapes the chances and draws one token. **Temperature** divides every logit first: below 1 it sharpens the
chances toward the top token, above 1 it flattens them, and 0 means always take the top token (**greedy**).
**Top-k** keeps only the k likeliest tokens. **Top-p** keeps the fewest tokens whose chances add up to p. Both then
rescale the kept chances to add up to 1. Here is the toy's row after "the" under each setting.

In [2]:
from medha.sampler import chances

logits = toy.next_logits([0, 1])  # after the end token, then "the"; only the last token matters
print("token:    ", words)
settings = [("T 1", 1.0, None, None), ("T 0.5", 0.5, None, None), ("T 2", 2.0, None, None),
            ("T 0", 0.0, None, None), ("top-k 2", 1.0, 2, None), ("top-p 0.6", 1.0, None, 0.6)]
for label, temperature, top_k, top_p in settings:
    row = chances(logits, temperature, top_k, top_p)
    print(f"{label:10}", [round(chance, 3) for chance in row.tolist()])

token:     ['<|endoftext|>', 'the', 'cat', 'sat', 'on', 'mat']
T 1        [0.125, 0.125, 0.25, 0.125, 0.125, 0.25]
T 0.5      [0.083, 0.083, 0.333, 0.083, 0.083, 0.333]
T 2        [0.146, 0.146, 0.207, 0.146, 0.146, 0.207]
T 0        [0.0, 0.0, 1.0, 0.0, 0.0, 0.0]
top-k 2    [0.0, 0.0, 0.5, 0.0, 0.0, 0.5]
top-p 0.6  [0.2, 0.0, 0.4, 0.0, 0.0, 0.4]


**What you see**

At T 1 the row is the counts with add-one: cat and mat `0.25`, every other token `0.125`. T 0.5 squares each
chance before rescaling, so cat and mat grow to `0.333`. T 2 takes the square root, so they shrink to `0.207`. T 0
gives everything to cat: on a tie, the first top token wins. Top-k 2 keeps cat and mat, at `0.5` each.

Top-p 0.6: cat and mat hold 0.5, short of 0.6, so it keeps one more token. Four tokens tie at 0.125, and the lowest
id, the end token, comes first. The three kept tokens rescale to `[0.2, 0.0, 0.4, 0.0, 0.0, 0.4]`.

The **decode loop** writes text: score the next token, pick one, append it, and go again. It stops when it picks
the end token, or when it reaches its limit of new tokens. A seeded random generator makes the draws repeatable.

In [3]:
import torch
from medha.sampler import generate

for label, temperature in [("seed 0, T 1:", 1.0), ("greedy, T 0:", 0.0)]:
    generator = torch.Generator().manual_seed(0)
    steps = generate(toy, [0], max_new_tokens=10, end=0, generator=generator, temperature=temperature)
    picked = []
    for step in steps:
        picked.append(f"{words[step.token]} ({step.chance:.2f})")
    print(label, " ".join(picked))

seed 0, T 1: cat (0.14) sat (0.29) mat (0.14) on (0.14) <|endoftext|> (0.14)
greedy, T 0: the (1.00) cat (1.00) sat (1.00) on (1.00) the (1.00) cat (1.00) sat (1.00) on (1.00) the (1.00) cat (1.00)


**What you see**

With seed 0 the loop starts after the end token and draws cat, chance `0.14`, though "the" had 0.29. It goes
on to sat, mat and on, then draws the end token, `<|endoftext|> (0.14)`, and stops. Only cat then sat is a pair from
the training story. The other four picks come from the counts that smoothing added: the tail that top-k and top-p
cut off.

Greedy always takes the top token. It writes `the (1.00) cat (1.00) sat (1.00) on (1.00) the` and goes round that
loop until it reaches its limit of 10 tokens.

Used in: `uv run vak write` prints the same table for the real model, then runs the same loop.

## Recap

| Idea | Remember | Used next |
|---|---|---|
| Sampler | temperature, then top-k, then top-p, then rescale to 1 | the GPT's chances go through the same `chances` |
| Decode loop | score, pick, append, repeat; stop at the end token or the limit | the GPT writes through the same `generate`; the KV cache makes each step cheaper |